In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:

def display_average(averages, stds, decimals=2):
    """
    Combine averages and standard deviations into the format:
    mean (std).

    Parameters
    ----------
    averages : pandas.DataFrame
        Mean values.

    stds : pandas.DataFrame
        Standard deviation values.

    decimals : int, default=3
        Number of decimal places.

    Returns
    -------
    pandas.DataFrame
        Formatted table.
    """
    result = averages.copy().astype(str)

    for col in averages.columns:
        result[col] = [
            f"{avg:.{decimals}f} ({sd:.{decimals}f})"
            for avg, sd in zip(
                averages[col],
                stds[col],
            )
        ]

    return result


def best_result(path, methods, metrics, extension=".txt", dp=3):
    """
    Extract the metric values corresponding to the
    minimum objective function value.

    Parameters
    ----------
    path : str or Path
        Directory containing the result files.

    methods : list of str
        Method names.

    metrics : list of str
        Metrics to extract.

    extension : str, default=".txt"
        Result file extension.

    dp : int, default=3
        Number of decimal places.

    Returns
    -------
    pandas.DataFrame
        Best metric values for each method.
    """
    path = Path(path)
    results = []
    for method in methods:
        file_path = path / f"{method}{extension}"
        df = pd.read_csv(file_path)
        idx_best = df["J"].idxmin()
        results.append(df.loc[idx_best, metrics].to_numpy())
    return pd.DataFrame(results, index=methods, columns=metrics).round(dp)



def average_result(path, methods, metrics, extension=".txt", dp=3):
    """
    Compute the mean and standard deviation of the
    selected metrics across runs.

    Parameters
    ----------
    path : str or Path
        Directory containing the result files.

    methods : list of str
        Method names.

    metrics : list of str
        Metrics to summarize.

    extension : str, default=".txt"
        Result file extension.

    dp : int, default=3
        Number of decimal places.

    Returns
    -------
    dict
        Dictionary containing:

        - 'rs' : formatted table.
        - 'average' : mean values.
        - 'std' : standard deviation values.
    """
    path = Path(path)
    averages = []
    stds = []
    for method in methods:
        file_path = path / f"{method}{extension}"
        df = pd.read_csv(file_path)

        averages.append(
            df[metrics].mean().to_numpy()
        )

        stds.append(
            df[metrics].std().to_numpy()
        )

    averages = pd.DataFrame(averages, index=methods, columns=metrics)
    stds = pd.DataFrame(stds, index=methods, columns=metrics)
    rs = display_average(averages, stds, decimals=dp )

    return {
        "rs": rs,
        "average": averages.round(dp),
        "std": stds.round(dp),
    }



def get_performance(path, methods, datasets, metric="ARI", mode="best"):

    """
    Extract the performance of a metric across datasets.

    Parameters
    ----------
    path : str or Path
        Directory containing the dataset folders.

    methods : list of str
        Method names.

    datasets : list of str
        Dataset names.

    metric : str, default="ARI"
        Metric to extract.

    mode : {"best", "average"}, default="best"
        Whether to use the best run or the average over runs.

    Returns
    -------
    pandas.DataFrame
        Performance table with methods as rows and datasets as columns.
    """

    if mode not in {"best", "average"}:
        raise ValueError(
            "mode must be either 'best' or 'average'."
        )

    path = Path(path)

    performance = pd.DataFrame(
        index=methods,
        columns=datasets,
        dtype=float,
    )

    for dataset in datasets:

        dataset_path = path / dataset

        if mode == "best":
            results = best_result(
                path=dataset_path,
                methods=methods,
                metrics=[metric],
            )

        else:
            results = average_result(
                path=dataset_path,
                methods=methods,
                metrics=[metric],
            )["average"]

        performance[dataset] = results.loc[methods,metric]

    return performance

In [3]:
datasets = ['car_models', 'fungi', 'horses', 'temperature', 'fish', 'abalone' ]
methods = ['IFCM', 'KFCM-IV', 'IFDK', 'IKFDK-O', 'IKFDK-K']

### Best Solution Analysis

##### ARI

In [4]:
path_results = path = '../benchmark_application/results/'
X = get_performance(path_results, methods=methods, datasets=datasets, metric='ARI', mode='best')
avr_result = np.round(X.mean(axis=1),3)
avr_ranking =  np.round(X.rank(axis=0, ascending=False,method="min").mean(axis=1),3)
df = X.astype('str')
df['---'] = '---'
df['avr_result'] = avr_result.astype('str')
df['avr_rank'] = avr_ranking.astype('str')
df.T



,IFCM,KFCM-IV,IFDK,IKFDK-O,IKFDK-K
car_models,0.279,0.279,0.279,0.365,0.354
fungi,0.396,0.339,0.022,0.311,0.422
horses,0.366,0.366,0.366,0.427,0.366
temperature,0.566,0.566,0.546,0.546,0.546
fish,-0.047,-0.047,-0.047,0.103,-0.061
abalone,0.02,0.02,0.02,0.02,0.193
---,---,---,---,---,---
avr_result,0.263,0.254,0.198,0.295,0.303
avr_rank,2.0,2.167,2.833,2.0,2.333


##### HUL

In [5]:
path_results = path = '../benchmark_application/results/'
X = get_performance(path_results, methods=methods, datasets=datasets, metric='HUL', mode='best')
avr_result = np.round(X.mean(axis=1),3)
avr_ranking =  np.round(X.rank(axis=0, ascending=False,method="min").mean(axis=1),3)
df = X.astype('str')
df['---'] = '---'
df['avr_result'] = avr_result.astype('str')
df['avr_rank'] = avr_ranking.astype('str')
df.T



,IFCM,KFCM-IV,IFDK,IKFDK-O,IKFDK-K
car_models,0.661,0.661,0.661,0.722,0.763
fungi,0.726,0.696,0.521,0.677,0.747
horses,0.788,0.788,0.788,0.803,0.788
temperature,0.793,0.793,0.785,0.785,0.789
fish,0.515,0.515,0.515,0.663,0.606
abalone,0.58,0.58,0.582,0.581,0.608
---,---,---,---,---,---
avr_result,0.677,0.672,0.642,0.705,0.717
avr_rank,2.5,2.667,3.167,2.5,1.667


### Average Analysis

##### ARI

In [6]:
path_results = path = '../benchmark_application/results/'
X = get_performance(path_results, methods=methods, datasets=datasets, metric='ARI', mode='average')
avr_result = np.round(X.mean(axis=1),3)
avr_ranking =  np.round(X.rank(axis=0, ascending=False,method="min").mean(axis=1),3)
df = X.astype('str')
df['---'] = '---'
df['avr_result'] = avr_result.astype('str')
df['avr_rank'] = avr_ranking.astype('str')
df.T



,IFCM,KFCM-IV,IFDK,IKFDK-O,IKFDK-K
car_models,0.338,0.339,0.293,0.391,0.355
fungi,0.232,0.281,0.036,0.171,0.093
horses,0.303,0.305,0.362,0.39,0.354
temperature,0.537,0.523,0.534,0.536,0.503
fish,-0.03,-0.024,-0.028,0.055,0.084
abalone,0.053,0.053,0.047,0.103,0.039
---,---,---,---,---,---
avr_result,0.239,0.246,0.207,0.274,0.238
avr_rank,3.167,2.833,3.833,1.667,3.333


##### HUL

In [7]:
path_results = path = '../benchmark_application/results/'
X = get_performance(path_results, methods=methods, datasets=datasets, metric='HUL', mode='average')
avr_result = np.round(X.mean(axis=1),3)
avr_ranking =  np.round(X.rank(axis=0, ascending=False,method="min").mean(axis=1),3)
df = X.astype('str')
df['---'] = '---'
df['avr_result'] = avr_result.astype('str')
df['avr_rank'] = avr_ranking.astype('str')
df.T



,IFCM,KFCM-IV,IFDK,IKFDK-O,IKFDK-K
car_models,0.74,0.742,0.682,0.715,0.763
fungi,0.637,0.668,0.529,0.609,0.574
horses,0.759,0.761,0.789,0.798,0.784
temperature,0.779,0.774,0.779,0.78,0.77
fish,0.611,0.629,0.588,0.651,0.683
abalone,0.585,0.585,0.588,0.594,0.585
---,---,---,---,---,---
avr_result,0.685,0.693,0.659,0.691,0.693
avr_rank,3.167,2.833,3.5,2.0,2.833
